# Ajax quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YannBerthelot/Ajax/blob/main/notebooks/quickstart.ipynb)

This notebook trains a first agent, reads its results, acts with the trained policy and saves it: the documentation's [quickstart](https://yannberthelot.github.io/Ajax/latest/getting-started/quickstart/), cell by cell. Each cell runs on a CPU in under a minute, compilation included.

In [ ]:
%pip install -q ajax-rl

## Train an agent

An agent takes an environment id and its hyperparameters, and `train` runs it. Here PPO learns CartPole with three seeds at once, and the plot shows their evaluations.

In [ ]:
import matplotlib.pyplot as plt

from ajax import PPO

agent = PPO("CartPole-v1", n_envs=8, n_steps=128)  # 1,024 steps per rollout
state, evaluations = agent.train(seed=[1, 2, 3], n_timesteps=40_960)

steps = evaluations.timesteps
returns = evaluations.returns  # shape (3, 10): seeds x evaluations

plt.plot(steps, returns.mean(axis=0))
plt.fill_between(steps, returns.min(axis=0), returns.max(axis=0), alpha=0.3)
plt.xlabel("environment steps")
plt.ylabel("evaluation return")
plt.show()

`train` compiled the training, environment included, and ran the three seeds side by side. Ten times over the run, it evaluated each seed's policy on 10 test episodes and printed their mean return. `evaluations` holds what was printed and every other logged value: a dict of each logged name to an array with a row per seed and a column per evaluation. `returns` and `timesteps` are the two most read.

To choose when the evaluations come, or to log to TensorBoard or Weights & Biases, pass a `LoggingConfig`; `logging_config=None` trains with no evaluation and prints nothing.

## Continuous actions

The same calls work for every agent. SAC on Pendulum, for a few thousand steps: enough to show the mechanics, not to train a pendulum.

In [ ]:
from ajax import SAC

# SAC waits learning_starts steps (10,000 by default) before its first update;
# this short demonstration starts after 1,000.
sac = SAC("Pendulum-v1", learning_starts=1_000)
_, sac_evaluations = sac.train(seed=[1, 2], n_timesteps=3_000)

## Use the trained policy

`evaluate` plays fresh episodes with each seed's trained policy and returns their returns, a row per seed.

In [ ]:
returns = agent.evaluate(state, episodes=20)  # shape (3, 20)
print(returns.mean(axis=1))  # each seed's score

To act in a loop of your own, `agent.predict(state, observation)` returns the first seed's action for an observation, in the environment's units ([Train, evaluate, deploy](https://yannberthelot.github.io/Ajax/latest/guide/deploy/)).

## Save and load

`agent.save` writes the state with the agent's class and arguments, and `ajax.load` builds both again.

In [ ]:
import os
import tempfile

import ajax

path = os.path.join(tempfile.mkdtemp(), "ppo_cartpole.pkl")
agent.save(state, path)

# Later, in another process.
agent, state = ajax.load(path)
print(agent.evaluate(state, episodes=5).mean(axis=1))

## Next

- [Why Ajax](https://yannberthelot.github.io/Ajax/latest/getting-started/why-ajax/): what else Ajax offers, and when another library fits better.
- [Coming from SB3](https://yannberthelot.github.io/Ajax/latest/getting-started/coming-from-sb3/): Stable-Baselines3's calls and defaults beside Ajax's.
- [Algorithms](https://yannberthelot.github.io/Ajax/latest/algorithms/): which agent for which problem, and each one's defaults.
- [Training an agent](https://yannberthelot.github.io/Ajax/latest/guide/training/): seeds, sweeps and resuming.